In [1]:
import json
import torch
from transformers import pipeline

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

line_name = "BIG EDDY-OSTRAND 500KV"

base_message = [
    {
        "role": "system",
        "content": """
You extract the two endpoint bus/substation names from BPA transmission-line names.

Return ONLY valid JSON using exactly this format:

{
  "first_bus": "BUS_NAME",
  "second_bus": "BUS_NAME"
}

Rules:
- Extract the two substations or buses connected by the transmission line.
- Remove voltage levels, circuit numbers, and other equipment information.
- Preserve the actual bus/substation names.
- Do not add explanations.
- Do not use Markdown.
- Do not guess.
- If either endpoint cannot be determined, return null for that endpoint.
""",
    },
]

/home/aj/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 254/254 [00:00<00:00, 338.82it/s]


In [ ]:
from transformers import logging

logging.set_verbosity_error()
pipe.generation_config.max_length = None
pipe.generation_config.max_new_tokens = 100
pipe.generation_config.do_sample = False
pipe.generation_config.temperature = None


def get_buses_from_line_name(line_name):
    prompt = base_message + [
        {"role": "user", "content": f"Transmission line name: {line_name}"}
    ]
    output = pipe(prompt)
    response = output[0]["generated_text"][-1]["content"]
    buses = json.loads(response)
    return buses["first_bus"], buses["second_bus"]

In [3]:
get_buses_from_line_name("BIG EDDY-OSTRAND 500KV")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


('BIG EDDY', 'OSTRAND')

In [ ]:
from pathlib import Path

import pandas as pd
from tqdm import tqdm

INPUT_PATH = Path("bpa_concatenated.csv")
OUTPUT_PATH = Path("bpa_processed.scv")
CHECKPOINT_PATH = Path("bpa_processed.checkpoint.csv")
CHECKPOINT_EVERY = 100
PROCESSED_COLUMN = "_Bus Extraction Complete"

if CHECKPOINT_PATH.exists():
    bpa = pd.read_csv(CHECKPOINT_PATH)
    print(f"Resuming from {CHECKPOINT_PATH.resolve()}")
else:
    bpa = pd.read_csv(INPUT_PATH)
    bpa["First Bus"] = pd.NA
    bpa["Second Bus"] = pd.NA
    bpa[PROCESSED_COLUMN] = False

bpa["First Bus"] = bpa["First Bus"].astype("object")
bpa["Second Bus"] = bpa["Second Bus"].astype("object")
completed = bpa[PROCESSED_COLUMN].astype(str).str.lower().eq("true")
pending_indices = bpa.index[~completed]
bus_cache = {
    row["Name"]: (row["First Bus"], row["Second Bus"])
    for _, row in bpa.loc[completed].iterrows()
    if row["First Bus"] != "---" and row["Second Bus"] != "---"
}
error_count = int(((bpa["First Bus"] == "---") & (bpa["Second Bus"] == "---")).sum())


def save_checkpoint():
    temporary_path = CHECKPOINT_PATH.with_suffix(".tmp")
    bpa.to_csv(temporary_path, index=False)
    temporary_path.replace(CHECKPOINT_PATH)


progress = tqdm(
    pending_indices,
    total=len(pending_indices),
    desc="Extracting line buses",
    unit="row",
)

try:
    for processed_count, row_index in enumerate(progress, start=1):
        line_name = bpa.at[row_index, "Name"]
        try:
            if line_name in bus_cache:
                first_bus, second_bus = bus_cache[line_name]
            else:
                first_bus, second_bus = get_buses_from_line_name(line_name)
                bus_cache[line_name] = (first_bus, second_bus)
        except Exception:
            first_bus = "---"
            second_bus = "---"
            error_count += 1
            progress.set_postfix(errors=error_count)

        bpa.at[row_index, "First Bus"] = first_bus
        bpa.at[row_index, "Second Bus"] = second_bus
        bpa.at[row_index, PROCESSED_COLUMN] = True

        if processed_count % CHECKPOINT_EVERY == 0:
            save_checkpoint()
finally:
    save_checkpoint()

bpa.drop(columns=PROCESSED_COLUMN).to_csv(OUTPUT_PATH, index=False)
print(
    f"Saved {len(bpa):,} rows to {OUTPUT_PATH.resolve()} "
    f"({error_count:,} extraction errors)"
)